### This notebook implements Deep Learning with CNNs. 

Specifically, we provide code for:

```
- training CNNs for individual mutational series (Figs 3B, C)
- training CNNs for aggregates of mutational series (SFig 4B)
- creating and training models for impact of diversity (Fig 4A)
```

**Note**: to enable the GPU (provided for free by Google), click the *Edit* option (tab at the top), then *Notebook Settings*, and finally pick *GPU* in the *Harware Accelerator* dropdown menu.

## 0. Mount drive and set the main path

In [ ]:
#mount to drive
from google.colab import drive
try:
  drive.mount('/content/drive', force_remount=True)
except:
  drive._mount('/content/drive', force_remount=True)

# construct the path to use
import os
 
# get the current working directory
dir_path = os.path.dirname(os.path.realpath('__file__'))

# loop to crawl over your Drive and construct the correct path
for root, dirs, files in os.walk(dir_path):
    for file in files:
 
        # do not change the extension *.ipynb*
        if file.endswith('1_kmers.ipynb'):
          # check that we're using the correct parent directory
          if f'{root}/{file}'[-24:] == '/seq2yield/1_kmers.ipynb':
            # create path to the import directory
            drive_dir = f"{f'{root}/{file}'[:-13]}to_import/"

drive_dir

## 1. Import libraries and get data 

In [ ]:
import os
import random
import pickle
from google.colab import files

import scipy
import scipy.stats
import pandas as pd
import numpy as np

!pip install pyyaml h5py

import sys
sys.path.insert(0, drive_dir)

import utils, cnn_conv2d, training_schemes

# LOAD .csv into a DataFrame
pd.options.display.max_rows = 10
raw_data = pd.read_csv(f'{drive_dir}Ecoli_data.csv', index_col= 0).dropna().reset_index(drop = True)

## 2. Generate or load sets
```
working_set: genotype-phenotype pairs to split for training in various percentages
heldout_set: genotype-phenotype pairs to use **only** for testing
```

In [ ]:
iter_ID   = 1 #choose an integer in [1,5] to load different sets
get_saved = True #set to *True* to use saved working/heldout sets; set to *False* to generate new sets

###
iteration = f'iteration_{iter_ID}/'
series_dict = utils.split_seeds(raw_data)

working_set, heldout_set = utils.get_split(series_dict = series_dict, 
                                           series_list = [sr for sr in range(1,57)],
                                           load_saved = get_saved,
                                           train_size = 0.9, #change to get different dataset ratios
                                           path = f'{drive_dir}_saved/{iteration}')

print(f'Working set contains {working_set.shape[0]} sequences for train and validation\nHeldout contains {heldout_set.shape[0]} sequences for testing')

## 3. Define hyperparameters for *deep* regressors (CNN)

```
Note (1): I provide a manual option to change hyperparameters if needed; else load from *.pkl*
Note (2): Hyperparameters only include architecture-related arguments
```


In [ ]:
choose_params_2D = 'manual'

# Conv2D opt_params
if choose_params_2D == 'manual':
  opt_params_conv2D = {'conv_dropout': 0.15, 
                       'conv_filters': 256, #change this hyperparameter to implement CNNs of varying width
                       'conv_layers': 3, #change this hyperparameter to implement CNNs of varying depth
                       'conv_width': 13, 
                       'dense_dropout': 0.1, 
                       'dense_layers': 4, 
                       'dense_units': 256}
else:
  !cp /content/drive/MyDrive/paper_code/hyperOpt/test_params_1.pkl ./
  pickle_off_2d = open('test_params_1.pkl', 'rb')
  opt_params_conv2D = pickle.load(pickle_off_2d)

print(f'\nConv2D hyperparameters: {opt_params_conv2D}')

# options 
pheno = 'Protein'  
train_sz = 0.25
stratify = True
test_indiv_series = True
to_print = False

batch = 64
batch_norm = False
learning_rate = 1e-3  
epochs = 100          
patience = 15       

###
model_config = dict({'batch' : batch,
                     'batch_norm' : batch_norm,
                     'learning_rate' : learning_rate,
                     'epochs' : epochs,
                     'patience' : patience,
                     'opt_params' : opt_params_conv2D,
                     'train_size' : train_sz,
                     'phenotype' : pheno,
                     'stratify' : stratify,
                     'saved_split' : False})

## 4. Train CNNs

In [ ]:
!mkdir -p training_entire
checkpoint_path = "training_entire/cp-{epoch:04d}.ckpt"
checkpoint_dir = os.path.dirname(checkpoint_path)

import tensorflow as tf
from sklearn.metrics import mean_squared_error, r2_score
from keras.callbacks import History, EarlyStopping, ModelCheckpoint

cfg = tf.compat.v1.ConfigProto()
cfg.gpu_options.allow_growth = True
with tf.compat.v1.Session(config=cfg) as sess:

  # define mutational series ID
  series_list = [sr for sr in range(1, 2)]

  # trains one CNN for a combination of mutational series if *True*
  # train one CNN per series if *False*
  aggregate = False

  # get training & validation sets (->dict)
  train_, val_ = training_schemes.split_sets(series_dict = utils.split_seeds(working_set),
                                             series_list = series_list,
                                             model_config = model_config,
                                             cumul = aggregate 
                                            )
  
  # build test dict from heldout_set
  heldout_set_dict = utils.split_seeds(heldout_set)
  if aggregate:
    test_ = {'cumul' : training_schemes.df_concat(dict_ = heldout_set_dict, lst_ = series_list)}
  else:
    test_ = {f'seed_{sr}':heldout_set_dict[f'seed_{sr}'].reset_index(drop=True) for sr in series_list}

  # init lists to hold R2 results
  results_train, results_val, results_test = [], [], []
  history_dicts_lst = []
  # train models
  for key in list(train_.keys()):
    r2_train, r2_val, r2_test, history_obj = training_schemes.cnn_predict(train_df = train_[key],
                                                                          val_df   = val_[key],
                                                                          test_obj  = test_[key],
                                                                          model_config = model_config,
                                                                          checkpoint_path = checkpoint_path
                                                                          )
    # store R2 score in equivalent list
    results_train.append(r2_train)
    results_val.append(r2_val)
    results_test.append(r2_test)
    history_dicts_lst.append(history_obj)

## 5. Impact of sequence diversity of model generalization

### 5.1 Load, or generate new, mutational series for 27 models

In [ ]:
itr_id, get_new = 1, True #get_new = True will generate a new combination of 27 models to train
LOAD_DIR = f'{drive_dir}_saved/iteration_{itr_id}/results/diversity/'

# define mutational series ID
series_list = [sr for sr in range(1, 57)]

if get_new:
  in_s, bw_s = training_schemes.get_diversity(generate_new=True)
else:
  in_s, bw_s = training_schemes.get_diversity(mut_idxs = training_schemes.load_diversity(iteration_ID = itr_id, load_path = LOAD_DIR))

exp_dict = training_schemes.diversity_sets(C = 5800,  #the total size of the dataset to train on
                                           model_config=model_config,
                                           series_dict = utils.split_seeds(working_set),
                                           heldout_set_dict = utils.split_seeds(heldout_set),
                                           in_series = in_s, 
                                           bw_series = bw_s,
                                           aggregate = False
                                          )

### 5.2 Training loop: 27 models

In [ ]:
!mkdir -p training_entire
checkpoint_path = "training_entire/cp-{epoch:04d}.ckpt"
checkpoint_dir = os.path.dirname(checkpoint_path)

import tensorflow as tf
from sklearn.metrics import mean_squared_error, r2_score
from keras.callbacks import History, EarlyStopping, ModelCheckpoint

cfg = tf.compat.v1.ConfigProto()
cfg.gpu_options.allow_growth = True
with tf.compat.v1.Session(config=cfg) as sess:

  # init results dataframe
  results_df = pd.DataFrame()

  # train and evaluate 27 models
  for exp in exp_dict.keys():
    r2_train, r2_val, r2_test, _ = training_schemes.cnn_predict(train_df = exp_dict[exp]['train']['cumul'],
                                                                val_df   = exp_dict[exp]['val']['cumul'],
                                                                test_obj  = exp_dict[exp]['test'],
                                                                model_config = model_config, 
                                                                test_1M_to_cumul = False, #evaluates 1 trained model in each mut. series -- do not change!
                                                                checkpoint_path = checkpoint_path
                                                                )
    # store R2 for each series in a results dataframe
    results_df[exp] = list(r2_test.values())
  # add a seeds column
  results_df['seed']  = [sr for sr in range(1, 57)]

  # display results dataframe
  display(results_df)